# Population: Sexes by age in Ireland

## Part 1: Differences between sexes by age in Ireland


In [42]:
# Import necessary libraries
import pandas as pd # For calculation and data manipulation
import matplotlib.pyplot as plt # For data visualization
import numpy as np    # For data processing

# Confirmation message
print("Libraries imported successfully!")


Libraries imported successfully!


In [43]:
# Define file path and name we will be looking at 
FILENAME = "cso_populationbyageandsex.csv"
DATADIR = "data/"
FULLPATH = DATADIR + FILENAME

# Read the CSV file into a Data Frame
df = pd.read_csv(FULLPATH)


In [44]:
# Drop all unnecessary columns
drop_col_list = ["Statistic Label","CensusYear","UNIT"]
df.drop(columns=drop_col_list, inplace=True)

**Before we get cracking with the task, we'll first edit some of the data, so we can perform the analysis easily.**


In [45]:
# Removing all ages
df = df[df["Single Year of Age"] != "All ages"]

# Changing the columns using a find and replace
df ["Single Year of Age"] = df ["Single Year of Age"].astype(str).str.replace("Under 1 year","0")
df ["Single Year of Age"] = df ["Single Year of Age"].astype(str).str.replace("100 years and over","100")

# Removing all non-digit characters from the selected column
df["Single Year of Age"] = df["Single Year of Age"].str.replace(r"\D", "", regex=True)

# Make sure the column is an integer for plotting later
df["Single Year of Age"] = df["Single Year of Age"].astype("int64")

# Remove Ireland as an Administrative County
df = df[df["Administrative Counties"] != "Ireland"]

# Check the general info of the dataframe 
print(df.info())

# Display the first and last 5 rows of data, to make sure the edits have worked
display(df.head())
display(df.tail())

# Source: Class notes
# Source: http://medium.com/@will4856/basic-steps-when-cleaning-a-data-set-using-pandas-3576e716173d


<class 'pandas.core.frame.DataFrame'>
Index: 9393 entries, 33 to 9791
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Sex                      9393 non-null   object
 1   Single Year of Age       9393 non-null   int64 
 2   Administrative Counties  9393 non-null   object
 3   VALUE                    9393 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 366.9+ KB
None


,Sex,Single Year of Age,Administrative Counties,VALUE
33,Both sexes,0,Carlow County Council,699
34,Both sexes,0,Dublin City Council,6213
35,Both sexes,0,Dún Laoghaire Rathdown County Council,2457
36,Both sexes,0,Fingal County Council,4009
37,Both sexes,0,South Dublin County Council,3544


,Sex,Single Year of Age,Administrative Counties,VALUE
9787,Female,100,Roscommon County Council,7
9788,Female,100,Sligo County Council,9
9789,Female,100,Cavan County Council,12
9790,Female,100,Donegal County Council,31
9791,Female,100,Monaghan County Council,7


In [46]:
# Creating a pivot table for population analysis by age and sex
df_analysis = pd.pivot_table(df, "VALUE", "Single Year of Age", "Sex")

# Saving the analysis to CSV file
# df_analysis.to_csv("population_by_age_and_sex.csv")

# Display the analysis
display(df_analysis)

Sex,Both sexes,Female,Male
Single Year of Age,,,
0,1864.387097,909.225806,955.161290
1,1820.000000,888.548387,931.451613
2,1910.000000,934.645161,975.354839
3,1951.096774,951.064516,1000.032258
4,1984.032258,961.903226,1022.129032
...,...,...,...
96,41.387097,30.838710,10.548387
97,30.612903,23.612903,7.000000
98,20.064516,15.870968,4.193548


*   ## **Weighted mean age (by sex)**


In [47]:
# We'll get some descriptive statistics for the population by age and sex in Ireland
# Using the existing df_analysis which contains population data by single year of age and sex
df_analysis.describe()

Sex,Both sexes,Female,Male
count,101.000000,101.000000,101.000000
mean,1644.566912,831.871607,812.695305
std,809.804432,399.652916,411.360302
min,14.225806,10.838710,3.387097
25%,1153.774194,590.129032,563.645161
50%,1949.516129,961.903226,975.290323
75%,2238.870968,1112.548387,1126.645161
max,2745.064516,1409.548387,1343.354839


In [48]:
def weighted_mean(df, single_year_of_age, sex, groupby):
    df = df.copy()
    grouped = df.groupby(groupby)
    df['weighted_average'] = df[single_year_of_age] / grouped[sex].transform('sum') * df[sex]
    return grouped['weighted_average'].sum(min_count=1) #min_count is required for Grouper objects

# Calculate mean ages by sex, REMOVING 'Both Sexes' from the results
mean_ages = weighted_mean(df[df['Sex'] != 'Both Sexes'], 'Single Year of Age', 'VALUE', 'Sex')
print("Mean ages by sex:")
print(mean_ages)

# Source: https://stackoverflow.com/questions/26205922/calculate-weighted-average-using-a-pandas-dataframe


Mean ages by sex:
Sex
Both sexes    38.346620
Female        38.939796
Male          37.739448
Name: weighted_average, dtype: float64


*   ## **The difference between the sexes by age - Not looking at regions**

In [49]:
# We need to focus on the differences in population by sex in Ireland, I am assuming all counties form Ireland,
# and the difference we are looking, is about the differences between the distribution of Males and Females, by age

# Use the existing df_analysis which already contains the Ireland-wide data by age and sex
pivot = df_analysis.copy()

# Reset index to make 'Single Year of Age' a column
pivot = pivot.reset_index()

# Check for differences between sexes in Ireland by age
pivot['Male-Female'] = pivot['Male'] - pivot['Female']
pivot['Sex Ratio (M/F)'] = pivot['Male'] / pivot['Female']

# Show the results for all ages
display(pivot[['Single Year of Age', 'Male', 'Female', 'Male-Female', 'Sex Ratio (M/F)']])

# Source: https://realpython.com/pandas-reset-index/
# Source: https://stackoverflow.com/questions/38951345/how-to-get-rid-of-multilevel-index-after-using-pivot-table-pandas


Sex,Single Year of Age,Male,Female,Male-Female,Sex Ratio (M/F)
0,0,955.161290,909.225806,45.935484,1.050522
1,1,931.451613,888.548387,42.903226,1.048285
2,2,975.354839,934.645161,40.709677,1.043556
3,3,1000.032258,951.064516,48.967742,1.051487
4,4,1022.129032,961.903226,60.225806,1.062611
...,...,...,...,...,...
96,96,10.548387,30.838710,-20.290323,0.342050
97,97,7.000000,23.612903,-16.612903,0.296448
98,98,4.193548,15.870968,-11.677419,0.264228
99,99,3.387097,10.838710,-7.451613,0.312500


# Part 2 - Making a variable that stores age 35 and groups the people within 5 years of that age together


In [30]:
# First, we'll group the data into age groups of 5 years each
bins = list(range(0, 105, 5))      # Create bins for every 5 years
labels = [f"{i}-{i+4}" for i in bins[:-1]]  # Create labels for the bins
# Assign age groups to a new column
df['Age Group'] = pd.cut(df['Single Year of Age'], bins=bins, labels=labels, right=False)
# Group by age group and sex, summing the population values for each group
age_grouped = df.groupby(['Age Group', 'Sex'])['VALUE'].sum().unstack(fill_value=0)

# Display the grouped data
display(age_grouped)

# Source: https://pandas.pydata.org/docs/reference/api/pandas.cut.html
# Source: https://www.geeksforgeeks.org/numpy/binning-data-in-python-with-scipy-numpy/

C:\Users\35387\AppData\Local\Temp\ipykernel_15000\2375995268.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_grouped = df.groupby(['Age Group', 'Sex'])['VALUE'].sum().unstack(fill_value=0)


Sex,Both sexes,Female,Male
Age Group,,,
0-4,295415,144007,151408
5-9,342670,167200,175470
10-14,374202,183088,191114
15-19,337628,165286,172342
20-24,307143,151697,155446
25-29,295808,148410,147398
30-34,332223,171706,160517
35-39,382869,199657,183212
40-44,411524,210963,200561


## Calculate the population difference between the sexes in that age group

In [31]:
# We'll show the data for the 35-39 age group
age_group = age_grouped.loc['35-39'] if '35-39' in age_grouped.index else None

display(age_group)

# And WE'll also show the ratio for Male and Females in the 35-39 age group
age_group_ratio = age_grouped.loc['35-39', 'Male'] / age_grouped.loc['35-39', 'Female'] if '35-39' in age_grouped.index else None

display(f'The ratio of Male to Female in the 35-39 age group is {age_group_ratio}')

# Source: https://www.geeksforgeeks.org/python/python-pandas-dataframe-loc/
# Source: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html

Sex
Both sexes    382869
Female        199657
Male          183212
Name: 35-39, dtype: int64

'The ratio of Male to Female in the 35-39 age group is 0.9176337418673024'

# Part 3 - Biggest population difference between the sexes in that age group

In [32]:
# Create a pivot table where it shows the biggest population difference between the sexes
# in that age group (35 to 39) and add the administrative counties

# Filter out 'Both Sexes' first, then group by Administrative Counties and Sex
clean_df = df[(df['Age Group'] == '35-39') & (df['Sex'].str.lower() != 'both sexes')]
county_group = clean_df.groupby(['Administrative Counties', 'Sex'])['VALUE'].sum().unstack(fill_value=0)

# Show the top 5 Administrative Counties with the biggest difference and remove 'Ireland' from the list
county_group = county_group[county_group.index != 'Ireland']
county_group['Difference'] = abs(county_group['Male'] - county_group['Female'])

# Display the top 5 counties with the biggest difference
display(county_group.sort_values(by='Difference', ascending=False).head(5))

# Source: https://www.geeksforgeeks.org/python/pandas-groupby-unstack/
# Source: https://pandas.pydata.org/pandas-docs/version/1.2.2/user_guide/reshaping.html
# Source: https://stackoverflow.com/questions/72495322/faster-alternative-to-groupby-unstack-then-fillna


Sex,Female,Male,Difference
Administrative Counties,,,
Fingal County Council,14415,12795,1620
Cork County Council,13355,11826,1529
South Dublin County Council,12797,11523,1274
Kildare County Council,10248,9266,982
Wicklow County Council,6017,5162,855


## Region With the Largest Difference

In [33]:
# Creating a pivot table with regions as rows showing individual ages 35-39 for each sex
# Filter data for ages 35-39
age_filtered = df[(df['Single Year of Age'] >= 35) & (df['Single Year of Age'] <= 39)]

# Group by Administrative Counties (removing Ireland), Single Year of Age
age_filtered = age_filtered[age_filtered['Administrative Counties'] != 'Ireland']
grouped = age_filtered.groupby(['Administrative Counties', 'Single Year of Age', 'Sex'])['VALUE'].sum().unstack(fill_value=0)

# Display the grouped data
display(grouped)

# Source: https://www.datacamp.com/de/tutorial/pandas-groupby (GroupBy)
# Source: https://stackoverflow.com/questions/69139030/why-and-when-should-use-a-stack-and-unstack-methods
# and https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.unstack.html (Unstack)

Sex                                         Both sexes  Female  Male
Administrative Counties Single Year of Age                          
Carlow County Council   35                         833     446   387
                        36                         835     438   397
                        37                         908     457   451
                        38                         948     483   465
                        39                        1006     553   453
...                                                ...     ...   ...
Wicklow County Council  35                        1983    1060   923
                        36                        2102    1153   949
                        37                        2267    1191  1076
                        38                        2354    1277  1077
                        39                        2473    1336  1137

[155 rows x 3 columns]

# END